# 1. Build the dataset
This builds the label-by-construction evaluation set (`PROTOCOL.md` §5). It selects MedQuAD passages, rewrites English answers, translates passages sentence by sentence (plus the answers) into Nigerian Pidgin and Hausa with Gemini, back-translates to filter, then builds items with gold reasoning traces.

**Before running:** add `GEMINI_API_KEY` in Colab Secrets (the key icon on the left) and turn on notebook access. No GPU is needed.

For `N_DOCS = 500`, expect roughly 400–500 API calls, at one call every 4 seconds. If the free tier's daily limit stops it, re-run the next day: every call is cached in Drive.

In [ ]:
REPO_URL = "https://github.com/KingQuinn/cross-lingual-groundedness-judge.git"
import os
if not os.path.exists("/content/repo"):
    !git clone -q $REPO_URL /content/repo
%cd /content/repo
!git pull -q
!pip -q install -r requirements.txt

In [ ]:
# Everything that costs API calls or GPU time is saved to Google Drive, so a
# disconnect never loses work; re-running a cell resumes where it stopped.
from google.colab import drive, userdata
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/cross-lingual-judge"
os.makedirs(WORK, exist_ok=True)

In [ ]:
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

## Unit tests (no API calls)

In [ ]:
!python -m pytest -q

## Build

In [ ]:
N_DOCS = 500
MODEL = "gemini-3.8-flash"   # any current Gemini Flash model your key can use
!python scripts/build_dataset.py --n_docs $N_DOCS --model $MODEL --batch_size 12 --out $WORK/data

In [ ]:
import json
s = json.load(open(f"{WORK}/data/stats.json"))
for k in ["n_passages", "rewrite", "translate_pcm", "translate_ha", "n_common_passages", "n_items", "n_robustness_variants"]:
    print(k, ":", s.get(k))
print(json.dumps(s["counts"]["ha<-en"], indent=1))

## Look at some gold reasoning traces
These are the fine-tuning targets. Read about ten of them in Pidgin and Hausa before going further.

In [ ]:
import random, sys
sys.path.insert(0, ".")
from judge.prompt import target_text
items = [json.loads(l) for l in open(f"{WORK}/data/items.jsonl")]
for it in random.Random(1).sample([i for i in items if i["source_lang"] == "en" and i["lang"] != "en"], 6):
    print(f"--- {it['lang']} | {it['item_type']}\nANSWER: {it['answer']}\n{target_text(it)}\n")

## Your checks (as a fluent reader)
1. Download `spotcheck_pcm.csv` and `spotcheck_ha.csv`, mark `y` or `n` per row for whether the meaning is preserved, and add them to `human_labels/`.
2. In `judge/text.py`, check the Hausa number words, the referral sentences, the condition and opener word lists, and `PIDGIN_VARIANTS`.

In [ ]:
from google.colab import files
for lang in ["pcm", "ha"]:
    files.download(f"{WORK}/data/spotcheck_{lang}.csv")

In [ ]:
!cd $WORK && zip -qr /content/data_to_commit.zip data -x "data/cache/*"
files.download("/content/data_to_commit.zip")